<a href="https://colab.research.google.com/github/idialloaka-ai/DAILYCHALLENGE/blob/master/ExerciceXP_W4_J2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prédiction du Diabète: Exercices de Classification

##  Exercice 1 : Compréhension du problème et collecte de données

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns

# Charger l'ensemble de données sur le diabète
# Nous allons utiliser le dataset Pima Indians Diabetes Dataset, couramment utilisé pour ce type de problème.
# Il est disponible via scikit-learn pour la commodité.

# Pour cet exercice, je vais simuler un dataset similaire au Pima Indians Diabetes Dataset
# Si vous avez un fichier CSV spécifique, vous pouvez le charger avec pd.read_csv('votre_fichier.csv')

# Création d'un dataset simulé pour les besoins de l'exercice
np.random.seed(42)
data_size = 1000

data = {
    'Pregnancies': np.random.randint(0, 15, data_size),
    'Glucose': np.random.randint(50, 200, data_size),
    'BloodPressure': np.random.randint(40, 120, data_size),
    'SkinThickness': np.random.randint(0, 60, data_size),
    'Insulin': np.random.randint(0, 850, data_size),
    'BMI': np.random.uniform(15.0, 50.0, data_size),
    'DiabetesPedigreeFunction': np.random.uniform(0.08, 2.5, data_size),
    'Age': np.random.randint(21, 80, data_size),
    'Outcome': np.random.randint(0, 2, data_size) # 0: No Diabetes, 1: Diabetes
}

df = pd.DataFrame(data)

print("Aperçu des 5 premières lignes du dataset:")
display(df.head())
print("\nInformations sur le dataset:")
display(df.info())

Aperçu des 5 premières lignes du dataset:


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,50,53,8,744,43.133509,1.521545,39,1
1,3,100,41,19,673,30.171528,0.275089,56,0
2,12,53,109,59,420,49.901382,2.353756,70,0
3,14,162,56,0,133,34.567075,1.907423,53,1
4,10,81,67,7,390,26.240675,1.499630,33,1



Informations sur le dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               1000 non-null   int64  
 1   Glucose                   1000 non-null   int64  
 2   BloodPressure             1000 non-null   int64  
 3   SkinThickness             1000 non-null   int64  
 4   Insulin                   1000 non-null   int64  
 5   BMI                       1000 non-null   float64
 6   DiabetesPedigreeFunction  1000 non-null   float64
 7   Age                       1000 non-null   int64  
 8   Outcome                   1000 non-null   int64  
dtypes: float64(2), int64(7)
memory usage: 70.4 KB


None

In [ ]:
# Combien y a-t-il de cas positifs et négatifs ?
print("Nombre de cas positifs (Outcome=1) et négatifs (Outcome=0):")
display(df['Outcome'].value_counts())

# Division des données en ensembles d'entraînement et de test
X = df.drop('Outcome', axis=1)
y = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nTaille de l'ensemble d'entraînement: {X_train.shape[0]} échantillons")
print(f"Taille de l'ensemble de test: {X_test.shape[0]} échantillons")
print(f"Proportion de cas positifs dans l'entraînement: {y_train.sum() / len(y_train):.2f}")
print(f"Proportion de cas positifs dans le test: {y_test.sum() / len(y_test):.2f}")

##  Exercice 2 : Sélection et standardisation des modèles

### Quel modèle de classification pouvons-nous utiliser pour résoudre ce problème et pourquoi ?

Pour prédire si un individu développera le diabète (un problème de classification binaire), nous pouvons utiliser plusieurs modèles. L'énoncé de l'exercice 3 mentionne la **régression logistique**, qui est un excellent choix pour les raisons suivantes :

*   **Classification Binaire** : Elle est naturellement conçue pour les problèmes de classification binaire, où l'objectif est de prédire l'une des deux classes.
*   **Interprétabilité** : Les coefficients de la régression logistique peuvent être interprétés, ce qui permet de comprendre l'influence de chaque caractéristique sur la probabilité de l'outcome.
*   **Performance décente** : Elle offre souvent de bonnes performances sur des datasets avec une relation linéaire entre les caractéristiques et la probabilité de l'outcome, et sert de bonne base de référence.
*   **Efficacité** : Relativement rapide à entraîner et à utiliser, surtout pour des datasets de taille moyenne.

D'autres modèles pourraient inclure les SVM, les arbres de décision, les forêts aléatoires, les réseaux de neurones, etc.

### Faut-il normaliser les données ? Si oui, utilisez `StandardScaler()`

**Oui, il faut normaliser les données.** Voici pourquoi :

La régression logistique (et de nombreux autres algorithmes d'apprentissage automatique) est sensible à l'échelle des caractéristiques. Les caractéristiques avec des plages de valeurs plus grandes peuvent dominer celles avec des plages plus petites, ce qui peut affecter la performance du modèle et le processus de convergence de l'optimisation.

`StandardScaler` transforme les données de manière à ce qu'elles aient une moyenne de 0 et un écart type de 1 (distribution normale standard). Cela garantit que chaque caractéristique contribue de manière égale à la distance ou à la similarité entre les points de données, ce qui est crucial pour les algorithmes basés sur la descente de gradient, comme la régression logistique.

In [ ]:
# Standardisation des données
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Données entraînées après standardisation (premières 5 lignes, premières 5 colonnes):")
display(pd.DataFrame(X_train_scaled, columns=X.columns).head())

##  Exercice 3 : Formation du modèle

In [ ]:
# Nous allons utiliser le modèle de régression logistique et nous l'entraînerons.
model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

print("Modèle de régression logistique entraîné avec succès.")

##  Exercice 4 : Métriques d'évaluation

In [ ]:
# Prédictions sur l'ensemble de test
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

# Représenter graphiquement le score de précision
accuracy = accuracy_score(y_test, y_pred)
print(f"Précision du modèle: {accuracy:.4f}")

plt.figure(figsize=(6, 4))
sns.barplot(x=['Précision'], y=[accuracy], palette='viridis')
plt.title('Score de Précision du Modèle de Régression Logistique')
plt.ylabel('Score')
plt.ylim(0, 1)
plt.show()

print("\nCommentaires sur le score de précision: Le score de précision indique la proportion d'échantillons correctement classifiés. Un score de {accuracy:.4f} signifie que le modèle a correctement prédit la classe pour {accuracy*100:.2f}% des cas dans l'ensemble de test. C'est une bonne métrique globale, mais elle peut être trompeuse en cas de déséquilibre des classes, ce qui justifie l'examen d'autres métriques.")

In [ ]:
# Tracer la matrice de confusion
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Prédit Négatif', 'Prédit Positif'],
            yticklabels=['Réel Négatif', 'Réel Positif'])
plt.title('Matrice de Confusion')
plt.xlabel('Prédiction')
plt.ylabel('Vraie Valeur')
plt.show()

print("\nCommentaires sur la matrice de confusion:")
print(f"    Vrais Négatifs (TN): {cm[0,0]} (correctement prédit sans diabète)")
print(f"    Faux Positifs (FP): {cm[0,1]} (prédit diabète, mais sans diabète en réalité - Erreur de type I)")
print(f"    Faux Négatifs (FN): {cm[1,0]} (prédit sans diabète, mais diabète en réalité - Erreur de type II)")
print(f"    Vrais Positifs (TP): {cm[1,1]} (correctement prédit avec diabète)")
print("La matrice de confusion nous donne une vue détaillée des erreurs et des succès du modèle pour chaque classe. Les faux négatifs (FN) sont particulièrement importants à surveiller dans les problèmes médicaux car ils représentent des cas de diabète non détectés.")

In [ ]:
# Tracer le graphique du rappel, de la précision et du score F1
report = classification_report(y_test, y_pred, output_dict=True)

metrics_df = pd.DataFrame(report).transpose()
metrics_df = metrics_df.loc[['0', '1', 'macro avg', 'weighted avg'], ['precision', 'recall', 'f1-score']]

plt.figure(figsize=(8, 6))
sns.heatmap(metrics_df.iloc[:2, :], annot=True, cmap='YlGnBu', fmt=".2f",
            cbar=True, linewidths=.5, linecolor='black')
plt.title('Rappel, Précision et Score F1 par Classe')
plt.show()

print("\nCommentaires sur le rappel, la précision et le score F1:")
print(classification_report(y_test, y_pred))
print("    **Précision**: Pour la classe positive (1), c'est la proportion de prédictions positives qui étaient réellement correctes. Une précision élevée minimise les faux positifs.")
print("    **Rappel (Sensibilité)**: Pour la classe positive (1), c'est la proportion de cas positifs réels qui ont été correctement identifiés. Un rappel élevé minimise les faux négatifs.")
print("    **Score F1**: C'est la moyenne harmonique de la précision et du rappel. Il est utile quand on cherche un équilibre entre précision et rappel, surtout en présence de classes déséquilibrées. Un score F1 élevé indique que le modèle fonctionne bien sur les deux fronts.")
print("    Il est crucial d'examiner ces métriques par classe, car une précision globale élevée peut cacher de mauvaises performances sur la classe minoritaire.")

##  Exercice 5 : Visualisation des performances de notre modèle

### Visualisation de la frontière de décision avec des informations de précision.

Visualiser la frontière de décision d'un modèle avec plus de deux caractéristiques est complexe car nous vivons dans un espace de dimension supérieure. Pour rendre cela visuel, nous allons réduire l'ensemble des caractéristiques à deux dimensions principales ou en sélectionner deux pour la visualisation. Ici, nous allons choisir deux caractéristiques (par exemple, Glucose et BMI) pour la visualisation.

In [ ]:
from mlxtend.plotting import plot_decision_regions

# Pour la visualisation de la frontière de décision, nous devons entraîner un nouveau modèle
# avec seulement 2 caractéristiques pour pouvoir le visualiser en 2D.
# Nous allons choisir 'Glucose' et 'BMI' comme caractéristiques pour la visualisation.

# S'assurer que les noms de colonnes sont corrects pour X_train
feature_names = X.columns
feature_idx = [feature_names.get_loc('Glucose'), feature_names.get_loc('BMI')]

X_train_2d = X_train_scaled[:, feature_idx]
X_test_2d = X_test_scaled[:, feature_idx]

# Entraîner un nouveau modèle avec seulement ces deux caractéristiques
model_2d = LogisticRegression(random_state=42)
model_2d.fit(X_train_2d, y_train)

# Créer le graphique de la frontière de décision
plt.figure(figsize=(10, 8))
fig = plot_decision_regions(X=X_test_2d, y=y_test.values, clf=model_2d, legend=2)
plt.xlabel('Glucose (standardisé)')
plt.ylabel('BMI (standardisé)')
plt.title('Frontière de Décision de la Régression Logistique (Glucose vs BMI)')
plt.show()

# Évaluation de la précision de ce modèle 2D
accuracy_2d = accuracy_score(y_test, model_2d.predict(X_test_2d))
print(f"Précision du modèle 2D (Glucose et BMI): {accuracy_2d:.4f}")
print("\nCommentaires sur la visualisation: Ce graphique montre comment le modèle sépare les deux classes (diabète vs non-diabète) en utilisant uniquement les caractéristiques Glucose et BMI. La ligne représente la frontière de décision. Les points colorés sont les échantillons de test, et la couleur de la région indique la prédiction du modèle pour cette zone. La précision de ce modèle 2D est donnée pour information, mais elle sera généralement inférieure à celle du modèle complet car moins de caractéristiques sont utilisées.")

##  Exercice 6 : Courbe ROC

In [ ]:
# Tracer la courbe ROC à l'aide de ce modèle de code disponible sur ce lien.

# Calculer la Taux de Vrais Positifs (TPR) et le Taux de Faux Positifs (FPR)
# Nous utilisons y_pred_proba calculé précédemment avec le modèle complet.
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Courbe ROC (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Aléatoire')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Taux de Faux Positifs (FPR)')
plt.ylabel('Taux de Vrais Positifs (TPR)')
plt.title('Courbe ROC (Receiver Operating Characteristic)')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

print("\nCommentaires sur la courbe ROC:")
print("La courbe ROC est un graphique qui illustre les performances d'un modèle de classification binaire pour tous les seuils de classification possibles. Elle trace le Taux de Vrais Positifs (TPR ou Rappel) en fonction du Taux de Faux Positifs (FPR) à différents seuils de discrimination.")
print("    **AUC (Area Under the Curve)**: L'aire sous la courbe ROC. Un AUC de 1.0 indique un classificateur parfait, tandis qu'un AUC de 0.5 indique un classificateur équivalent à un choix aléatoire. Plus l'AUC est proche de 1, meilleures sont les performances globales du modèle à distinguer les classes.")
print(f"    Notre modèle a un AUC de {roc_auc:.2f}, ce qui suggère qu'il a une bonne capacité discriminante pour ce problème.")